## Former code

In [ ]:
import numpy as np
import h5py
from astropy.io import fits
from astropy.table import Table
import pandas as pd

In [ ]:
# This cell converts .fits data file from COSMOS-Web into .csv for later use

master_path= "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    hdu.info()
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_laura.csv"
cosmos_cat.to_csv(output_path, index=False)


In [ ]:
data_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\"
cosmos_cat = pd.read_csv(data_path+"COSMOSWeb_laura.csv") # Data from COSMOS-WEB, converted from .fits to .csv in florah_eval_SFR.ipynb

I the next cell I will select the columns I want, from each extension selected before. This columns will be later saved in a new csv

In [ ]:
# Select the columns I want
columns = ['id', 'radius_sersic', 'ra', 'dec', 'sersic', 'sfr_med', 'mass_med', 'zpdf_med', 'sfr_inst', 'mass', 'morph_flag_f444w', 'b/t_f444w'  ]
cosmos_cat_filtered = cosmos_cat[columns]

# Save new csv
cosmos_cat_filtered.to_csv("C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_filtered.csv", index=False)


## New code:

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck15
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)



# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'radius_sersic': 'radius_sersic',
    'ra': 'ra',
    'dec': 'dec',
    'sersic': 'sersic',
    'zpdf_med': 'zpdf_med',
    'sfr_inst': 'sfr_raw',
    'mass': 'mass_raw',
    'morph_flag_f444w': 'morphology',
    'b/t_f444w': 'bovert'
}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 

# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')



# 3. Logarithmic transformations and physical filters
# Filter impossible values before applying logarithms
cosmos_cat = cosmos_cat[(cosmos_cat['mass_raw'] > 0) & (cosmos_cat['sfr_raw'] > 0)]

cosmos_cat['mass_CIGALE'] = np.log10(cosmos_cat['mass_raw'])
cosmos_cat['sfr_CIGALE'] = np.log10(cosmos_cat['sfr_raw'])



# 4. Pre-calculation - A Phase
# This helps relieve later usage of data, so that astrophysical calculations, such as angular
# diamater distance, that will be once calculated and stored in the output file.
# NOTE: This could compromise RAM usage, but might speed up code
z_array = cosmos_cat['zpdf_med'].values
dist_mpc = Planck15.angular_diameter_distance(z_array).value # Resultado en Mpc


# Calculate physical radius in log10(kpc)
# Formula: Physical_radius = Angular_diameter * Angular_aperture(rad)
cosmos_cat['log_radius_kpc'] = np.log10(dist_mpc * np.deg2rad(cosmos_cat['radius_sersic']) * 1e3)



# 5. Store cleaned file
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_processed.csv"
# Delete rows that contain NaNs in important columns
cosmos_cat.dropna(subset=['mass_CIGALE', 'sfr_CIGALE', 'zpdf_med', 'log_radius_kpc'], inplace=True)

cosmos_cat.to_csv(output_path, index=False)
print(f"Catálogo procesado guardado con {len(cosmos_cat)} filas.")

Catálogo procesado guardado con 588536 filas.
